In [1]:
import pandas as pd 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse 

In [2]:
import numpy as np

class MF(object):
    """docstring for CF"""
    def __init__(self, Y_data, K, lam = 0.1, Xinit = None, Winit = None, 
            learning_rate = 0.5, max_iter = 1000, print_every = 100, user_based = 1):
        self.Y_raw_data = Y_data
        self.K = K
        # regularization parameter
        self.lam = lam
        # learning rate for gradient descent
        self.learning_rate = learning_rate
        # maximum number of iterations
        self.max_iter = max_iter
        # print results after print_every iterations
        self.print_every = print_every
        # user-based or item-based
        self.user_based = user_based
        # number of users, items, and ratings. Remember to add 1 since id starts from 0
        self.n_users = int(np.max(Y_data[:, 0])) + 1 
        self.n_items = int(np.max(Y_data[:, 1])) + 1
        self.n_ratings = Y_data.shape[0]
        
        if Xinit is None: # new
            self.X = np.random.randn(self.n_items, K)
        else: # or from saved data
            self.X = Xinit 
        
        if Winit is None: 
            self.W = np.random.randn(K, self.n_users)
        else: # from saved data
            self.W = Winit
            
        # normalized data, update later in normalized_Y function
        self.Y_data_n = self.Y_raw_data.copy()


    def normalize_Y(self):
        if self.user_based:
            user_col = 0
            item_col = 1
            n_objects = self.n_users

        # if we want to normalize based on item, just switch first two columns of data
        else: # item based
            user_col = 1
            item_col = 0 
            n_objects = self.n_items

        users = self.Y_raw_data[:, user_col] 
        self.mu = np.zeros((n_objects,))
        for n in range(n_objects):
            # row indices of rating done by user n
            # since indices need to be integers, we need to convert
            ids = np.where(users == n)[0].astype(np.int32)
            # indices of all ratings associated with user n
            item_ids = self.Y_data_n[ids, item_col] 
            # and the corresponding ratings 
            ratings = self.Y_data_n[ids, 2]
            # take mean
            m = np.mean(ratings) 
            if np.isnan(m):
                m = 0 # to avoid empty array and nan value
            self.mu[n] = m
            # normalize
            self.Y_data_n[ids, 2] = ratings - self.mu[n]

    def loss(self):
        L = 0 
        for i in range(self.n_ratings):
            # user, item, rating
            n, m, rate = int(self.Y_data_n[i, 0]), int(self.Y_data_n[i, 1]), self.Y_data_n[i, 2]
            L += 0.5*(rate - self.X[m, :].dot(self.W[:, n]))**2
        
        # take average
        L /= self.n_ratings
        # regularization, don't ever forget this 
        L += 0.5*self.lam*(np.linalg.norm(self.X, 'fro') + np.linalg.norm(self.W, 'fro'))
        return L 
    
    def get_items_rated_by_user(self, user_id):
        """
        get all items which are rated by user user_id, and the corresponding ratings
        """
        ids = np.where(self.Y_data_n[:,0] == user_id)[0] 
        item_ids = self.Y_data_n[ids, 1].astype(np.int32) # indices need to be integers
        ratings = self.Y_data_n[ids, 2]
        return (item_ids, ratings)
        
        
    def get_users_who_rate_item(self, item_id):
        """
        get all users who rated item item_id and get the corresponding ratings
        """
        ids = np.where(self.Y_data_n[:,1] == item_id)[0] 
        user_ids = self.Y_data_n[ids, 0].astype(np.int32)
        ratings = self.Y_data_n[ids, 2]
        return (user_ids, ratings)
    
    def updateX(self):
        for m in range(self.n_items):
            user_ids, ratings = self.get_users_who_rate_item(m)
            Wm = self.W[:, user_ids]
            # gradient
            grad_xm = -(ratings - self.X[m, :].dot(Wm)).dot(Wm.T)/self.n_ratings + \
                                               self.lam*self.X[m, :]
            self.X[m, :] -= self.learning_rate*grad_xm.reshape((self.K,))
    
    def updateW(self):
        for n in range(self.n_users):
            item_ids, ratings = self.get_items_rated_by_user(n)
            Xn = self.X[item_ids, :]
            # gradient
            grad_wn = -Xn.T.dot(ratings - Xn.dot(self.W[:, n]))/self.n_ratings + \
                        self.lam*self.W[:, n]
            self.W[:, n] -= self.learning_rate*grad_wn.reshape((self.K,))

    def fit(self):
        self.normalize_Y()
        for it in range(self.max_iter):
            self.updateX()
            self.updateW()
            if (it + 1) % self.print_every == 0:
                rmse_train = self.evaluate_RMSE(self.Y_raw_data)
                print('iter =', it + 1, ', loss =', self.loss(), ', RMSE train =', rmse_train)

    def pred(self, u, i):
        """ 
        predict the rating of user u for item i 
        """
        u = int(u)
        i = int(i)
        if self.user_based:
            bias = self.mu[u]
        else: 
            bias = self.mu[i]
        pred = self.X[i, :].dot(self.W[:, u]) + bias 
        # truncate if results are out of range [0, 5]
        if pred < 0:
            return 0 
        if pred > 5: 
            return 5 
        return pred 
        
    
    def pred_for_user(self, user_id):
        """
        predict ratings one user give all unrated items
        """
        ids = np.where(self.Y_data_n[:, 0] == user_id)[0]
        items_rated_by_u = self.Y_data_n[ids, 1].tolist()              
        
        y_pred = self.X.dot(self.W[:, user_id]) + self.mu[user_id]
        predicted_ratings= []
        for i in range(self.n_items):
            if i not in items_rated_by_u:
                predicted_ratings.append((i, y_pred[i]))
        
        return predicted_ratings
    
    def evaluate_RMSE(self, rate_test):
        n_tests = rate_test.shape[0]
        SE = 0 # squared error
        for n in range(n_tests):
            pred = self.pred(rate_test[n, 0], rate_test[n, 1])
            SE += (pred - rate_test[n, 2])**2 

        RMSE = np.sqrt(SE/n_tests)
        return RMSE

## Input data

In [3]:
## Utilize schema from readme
u_data_cols = ['user_id', 'item_id', 'rating', 'timestamp']
u_item_cols = ['item_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
u_user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

In [4]:
df_users = pd.read_csv('data/u.user', sep='|', names=u_user_cols, encoding='latin-1')
df_items = pd.read_csv('data/u.item', sep='|', names=u_item_cols, encoding='latin-1')
df_ratings_base = pd.read_csv('data/ua.base', sep='\t', names=u_data_cols, encoding='latin-1')
df_ratings_test = pd.read_csv('data/ua.test', sep='\t', names=u_data_cols, encoding='latin-1')

In [5]:
rate_train = df_ratings_base.values
rate_test = df_ratings_test.values

# indices start from 0
rate_train[:, :2] -= 1
rate_test[:, :2] -= 1

In [6]:
rs = MF(rate_train, K = 10, lam = .1, print_every = 10, 
    learning_rate = 0.75, max_iter = 100, user_based = 1)
rs.fit()
# evaluate on test data
RMSE = rs.evaluate_RMSE(rate_test)
print('\nUser-based MF, RMSE =', RMSE)

iter = 10 , loss = 5.625790786460637 , RMSE train = 1.2126037178762898
iter = 20 , loss = 2.6299571834154905 , RMSE train = 1.04066882692033
iter = 30 , loss = 1.3416747699597809 , RMSE train = 1.0314492598212135
iter = 40 , loss = 0.7548284223599901 , RMSE train = 1.0309324412518897
iter = 50 , loss = 0.48591479996446013 , RMSE train = 1.0308882073659524
iter = 60 , loss = 0.36261046795926616 , RMSE train = 1.0308820221071717
iter = 70 , loss = 0.30606719585229014 , RMSE train = 1.0308809152894747
iter = 80 , loss = 0.2801378668808634 , RMSE train = 1.0308807028995595
iter = 90 , loss = 0.26824725548876205 , RMSE train = 1.0308806616158654
iter = 100 , loss = 0.26279447240023673 , RMSE train = 1.0308806536051802

User-based MF, RMSE = 1.0431349555781269


In [7]:
rs = MF(rate_train, K = 10, lam = .1, print_every = 10, learning_rate = 0.75, max_iter = 100, user_based = 0)
rs.fit()
# evaluate on test data
RMSE = rs.evaluate_RMSE(rate_test)
print('\nItem-based MF, RMSE =', RMSE)

c:\Users\vncpyy7h\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\vncpyy7h\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


iter = 10 , loss = 5.658752794726101 , RMSE train = 1.1791976430909377
iter = 20 , loss = 2.632527094630977 , RMSE train = 1.005716710793673
iter = 30 , loss = 1.3319522314742034 , RMSE train = 0.9970778871135675
iter = 40 , loss = 0.7393418792917329 , RMSE train = 0.9967825059409208
iter = 50 , loss = 0.467735158841432 , RMSE train = 0.996789151725119
iter = 60 , loss = 0.3431832681842705 , RMSE train = 0.9967938503328243
iter = 70 , loss = 0.28606467944889835 , RMSE train = 0.9967950392006232
iter = 80 , loss = 0.2598706021831912 , RMSE train = 0.996795309616153
iter = 90 , loss = 0.247858277978193 , RMSE train = 0.9967953698090484
iter = 100 , loss = 0.24234956492130755 , RMSE train = 0.996795383124136

Item-based MF, RMSE = 1.042652623120592


In [8]:
rs = MF(rate_train, K = 2, lam = 0, print_every = 10, learning_rate = 1, max_iter = 100, user_based = 0)
rs.fit()
# evaluate on test data
RMSE = rs.evaluate_RMSE(rate_test)
print('\nItem-based MF, RMSE =', RMSE)

iter = 10 , loss = 1.1715499150858444 , RMSE train = 1.480817820479227
iter = 20 , loss = 1.1056732808313043 , RMSE train = 1.46074736717496
iter = 30 , loss = 1.0471455550927633 , RMSE train = 1.4419444763520552
iter = 40 , loss = 0.994876440509565 , RMSE train = 1.4242853521541652
iter = 50 , loss = 0.9479738194003319 , RMSE train = 1.4076048944981119
iter = 60 , loss = 0.9057016406652127 , RMSE train = 1.3918701522334065
iter = 70 , loss = 0.8674479573408885 , RMSE train = 1.3770118306329475
iter = 80 , loss = 0.8327003978302946 , RMSE train = 1.3630212798686632
iter = 90 , loss = 0.8010271487957308 , RMSE train = 1.3497848277338156
iter = 100 , loss = 0.7720620714918958 , RMSE train = 1.3372726809418323

Item-based MF, RMSE = 1.4027515769702144
